<a href="https://colab.research.google.com/github/477117/Projeto_Anac/blob/main/Tarefa_1_Regras_de_Associacao_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercício: Regras de Associação (Apriori)

Neste notebook você irá praticar dois algoritmos importantes de **mineração de dados** e **aprendizado de máquina**:

* **Regras de associação (Apriori)** – usado em *Market Basket Analysis* para descobrir combinações de itens que ocorrem com frequência em transações comerciais.


### Conjuntos de dados utilizados

* **Groceries** – Conjunto de dados contendo **9 835** transações de um supermercado e **169** itens diferentes. As transações cobrem um período de 30 dias. Cada transação é uma lista de produtos comprados em um carrinho.



## 1 – Preparação do ambiente

Para extrair regras de associação com o Apriori usaremos a biblioteca `mlxtend`. Caso ela ainda não esteja instalada no ambiente do Colab, execute o comando abaixo para instalá‑la. Em seguida importaremos as bibliotecas necessárias.


In [36]:
import warnings

warnings.filterwarnings("ignore")


# Instala o mlxtend apenas se não estiver disponível
try:
    import mlxtend
except ImportError:
    !pip install -q mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações gerais de exibição
pd.set_option('display.max_columns', None)



## 2 – Análise de regras de associação

### 2.1 Carregar e explorar o dataset de compras (Groceries)

Vamos carregar o conjunto de dados **Groceries** a partir de um repositório público do GitHub. Cada linha representa uma transação e os produtos comprados são separados por vírgulas.


In [37]:

# Baixa o arquivo CSV diretamente do GitHub (sem cabeçalhos)
url_groceries = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv'

# Lê o arquivo como texto puro (uma transação por linha)

groceries_df = pd.read_csv(

    url_groceries,
    header=None,
    names=['items'],
    sep='|'

)

# Visualiza as primeiras linhas

groceries_df.head()

# Visualiza as primeiras transações
groceries_df.head()


,items
0,"citrus fruit,semi-finished bread,margarine,rea..."
1,"tropical fruit,yogurt,coffee"
2,whole milk
3,"pip fruit,yogurt,cream cheese ,meat spreads"
4,"other vegetables,whole milk,condensed milk,lon..."


In [38]:
transactions = groceries_df['items'].apply(
    lambda x: [item.strip() for item in x.split(',')]
)

In [6]:
transactions

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,items
0,"[citrus fruit, semi-finished bread, margarine,..."
1,"[tropical fruit, yogurt, coffee]"
2,[whole milk]
3,"[pip fruit, yogurt, cream cheese, meat spreads]"
4,"[other vegetables, whole milk, condensed milk,..."
...,...
9830,"[sausage, chicken, beef, hamburger meat, citru..."
9831,[cooking chocolate]
9832,"[chicken, citrus fruit, other vegetables, butt..."
9833,"[semi-finished bread, bottled water, soda, bot..."


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [39]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions_encoded = pd.DataFrame(te_ary, columns=te.columns_)
colunas_dummies = [col for col in df_transactions_encoded.columns]
df_transactions_encoded.sample(5)

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,bottled water,brandy,brown bread,butter,butter milk,cake bar,candles,candy,canned beer,canned fish,canned fruit,canned vegetables,cat food,cereals,chewing gum,chicken,chocolate,chocolate marshmallow,citrus fruit,cleaner,cling film/bags,cocoa drinks,coffee,condensed milk,cooking chocolate,cookware,cream,cream cheese,curd,curd cheese,decalcifier,dental care,dessert,detergent,dish cleaner,dishes,dog food,domestic eggs,female sanitary products,finished products,fish,flour,flower (seeds),flower soil/fertilizer,frankfurter,frozen chicken,frozen dessert,frozen fish,frozen fruits,frozen meals,frozen potato products,frozen vegetables,fruit/vegetable juice,grapes,hair spray,ham,hamburger meat,hard cheese,herbs,honey,house keeping products,hygiene articles,ice cream,instant coffee,jam,ketchup,kitchen towels,kitchen utensil,light bulbs,liqueur,liquor,liquor (appetizer),liver loaf,long life bakery product,make up remover,male cosmetics,margarine,mayonnaise,meat,meat spreads,misc. beverages,mustard,napkins,newspapers,nut snack,nuts/prunes,oil,onions,organic products,organic sausage,other vegetables,packaged fruit/vegetables,pasta,pastry,pet care,photo/film,pickled vegetables,pip fruit,popcorn,pork,pot plants,potato products,preservation products,processed cheese,prosecco,pudding powder,ready soups,red/blush wine,rice,roll products,rolls/buns,root vegetables,rubbing alcohol,rum,salad dressing,salt,salty snack,sauces,sausage,seasonal products,semi-finished bread,shopping bags,skin care,sliced cheese,snack products,soap,soda,soft cheese,softener,sound storage medium,soups,sparkling wine,specialty bar,specialty cheese,specialty chocolate,specialty fat,specialty vegetables,spices,spread cheese,sugar,sweet spreads,syrup,tea,tidbits,toilet cleaner,tropical fruit,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
1357,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
878,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False

# 3 - Gerar os conjuntos frequentes (frequent itemsets).

In [40]:
frequent_itemsets = apriori(df_transactions_encoded, min_support=0.01, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.033452,(UHT-milk)
1,0.017692,(baking powder)
2,0.052466,(beef)
3,0.033249,(berries)
4,0.026029,(beverages)
...,...,...
328,0.011998,"(root vegetables, whole milk, tropical fruit)"
329,0.014540,"(root vegetables, whole milk, yogurt)"
330,0.010473,"(whole milk, soda, yogurt)"
331,0.015150,"(whole milk, yogurt, tropical fruit)"


# 4 - Extrair regras de associação utilizando métricas como:
suporte (support)
confiança (confidence)
lift

In [48]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.1)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(beef),(other vegetables),0.052466,0.193493,0.019725,0.375969,1.943066,1.0,0.009574,1.292416,0.512224,0.087191,0.226255,0.238957
1,(other vegetables),(beef),0.193493,0.052466,0.019725,0.101944,1.943066,1.0,0.009574,1.055095,0.601792,0.087191,0.052218,0.238957
2,(beef),(rolls/buns),0.052466,0.183935,0.013625,0.259690,1.411858,1.0,0.003975,1.102329,0.307866,0.061159,0.092830,0.166882
3,(rolls/buns),(beef),0.183935,0.052466,0.013625,0.074074,1.411858,1.0,0.003975,1.023337,0.357463,0.061159,0.022805,0.166882
4,(root vegetables),(beef),0.108998,0.052466,0.017387,0.159515,3.040367,1.0,0.011668,1.127366,0.753189,0.120677,0.112977,0.245455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613,"(whole milk, yogurt)",(whipped/sour cream),0.056024,0.071683,0.010880,0.194192,2.709053,1.0,0.006864,1.152033,0.668309,0.093124,0.131970,0.172983
614,"(whipped/sour cream, yogurt)",(whole milk),0.020742,0.255516,0.010880,0.524510,2.052747,1.0,0.005580,1.565719,0.523711,0.040996,0.361316,0.283544
615,(whole milk),"(whipped/sour cream, yogurt)",0.255516,0.020742,0.010880,0.042579,2.052747,1.0,0.005580,1.022807,0.688864,0.040996,0.022299,0.283544
616,(whipped/sour cream),"(whole milk, yogurt)",0.071683,0.056024,0.010880,0.151773,2.709053,1.0,0.006864,1.112881,0.679582,0.093124,0.101431,0.172983


# 5 - Gerando 10 resultados encontrados por ordem de Lift

In [49]:
print("As 10 principais regras de associação (ordenadas por Lift):")
top_10_lift_rules = rules.sort_values(by="lift", ascending=False).head(10)
display(top_10_lift_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

As 10 principais regras de associação (ordenadas por Lift):


,antecedents,consequents,support,confidence,lift
458,"(whole milk, yogurt)",(curd),0.010066,0.179673,3.372304
459,(curd),"(whole milk, yogurt)",0.010066,0.188931,3.372304
440,"(other vegetables, citrus fruit)",(root vegetables),0.010371,0.359155,3.295045
441,(root vegetables),"(other vegetables, citrus fruit)",0.010371,0.095149,3.295045
559,"(yogurt, other vegetables)",(whipped/sour cream),0.010168,0.234192,3.267062
562,(whipped/sour cream),"(yogurt, other vegetables)",0.010168,0.141844,3.267062
512,"(tropical fruit, other vegetables)",(root vegetables),0.012303,0.342776,3.144780
513,(root vegetables),"(tropical fruit, other vegetables)",0.012303,0.112873,3.144780
5,(beef),(root vegetables),0.017387,0.331395,3.040367
4,(root vegetables),(beef),0.017387,0.159515,3.040367
